In [1]:
### Extract point values from S1 and S2

import numpy as np
from matplotlib import pyplot as plt

import shapefile as shp
from shapely.geometry import Point
from shapely.geometry import shape 
import pandas as pd
import matplotlib
import os
from datetime import datetime, timedelta
#import mpld3
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from scipy.stats import linregress
import matplotlib.font_manager as fm
from collections import Counter
import seaborn as sns
from osgeo import gdal
import subprocess
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd
from rasterio.mask import mask
from rasterio.plot import show
from osgeo import gdal, gdalconst, ogr
import glob
from scipy.ndimage import zoom
from rasterio.windows import Window
from rasterio.transform import Affine
from rasterio.merge import merge
from rasterio.sample import sample_gen

In [2]:
##### Read point coordinates ###
shapefile_path = "D:/Paul/YoloBypass/Shapefile/WaterPoints_20210228.shp"
gdf = gpd.read_file(shapefile_path)
raster_folder = 'D:/Paul/YoloBypass/Shapefile/optical/2021/'

In [3]:
files = os.listdir(raster_folder)

# Filter raster files based on their extensions
raster_files = [file for file in files if file.endswith(('.tif', '.tiff', '.img', '.jp2', '.png'))]

# Get the number of raster files
num_raster_files = len(raster_files)
num_raster_files

14

In [ ]:
data = []
# Open the raster file
# for raster_file in os.listdir(raster_folder):
#     raster_file_path = os.path.join(raster_folder, raster_file)
#     # Open the raster file
#     with rasterio.open(raster_file_path) as src:
#         # Create an empty list to store x, y, and pixel values for this raster file
#         subdata = []
#         # Loop through each point in the GeoDataFrame
#         for index, row in gdf.iterrows():
#             # Extract the point coordinates
#             point_coords = row.geometry.coords[0]
#             # Sample pixel value corresponding to the point
#             # This assumes the raster and shapefile are in the same CRS
#             # If not, you may need to reproject one of them
#             for val in sample_gen(src, [point_coords]):
#                 # Append ID, x, y, and pixel value to the subdata list
#                 subdata.append([index, point_coords[0], point_coords[1], val[0]])
#         # Store the subdata for this raster file in the data dictionary
#         data.extend(subdata)


In [4]:
data = []
for raster_file in os.listdir(raster_folder):
    raster_file_path = os.path.join(raster_folder, raster_file)
    with rasterio.open(raster_file_path) as src:
        subdata = []
        # Loop through each geometry in the shapefile
        for index, row in gdf.iterrows():
            # Extract the geometry of the feature
            geom = row.geometry
            # Loop through each point in the geometry
            for point in geom.geoms:
                # Sample pixel values corresponding to each point
                # This assumes the raster and shapefile are in the same CRS
                # If not, you may need to reproject one of them
                for val, point_coords in zip(sample_gen(src, [point.coords[0]]), point.coords):
                    # Append ID, x, y, and pixel value to the list
                    subdata.append([index, point_coords[0], point_coords[1], val[0]])  # Access th
    data.extend(subdata)

In [5]:
data = np.array(data)
data.shape
pxnb = data.shape[0]/14
pxnb = int(pxnb)

last_columns = []

for i in range(pxnb, len(data), pxnb):
    # Extract the last column from the current set of 108 elements
    last_column = data[i:i+pxnb, -1]
    # Append the last column to the last_columns list
    last_columns.append(last_column)

last_columns = np.array(last_columns)
last_columns = np.transpose(last_columns)
first = data[:pxnb, :]
conca = np.concatenate((first, last_columns), axis=1)

In [6]:
conca.shape

(125, 17)

In [8]:
csv_file_path = "D:/Paul/YoloBypass/Shapefile/2021_water_optical.csv"

# Write data_array to a CSV file
np.savetxt(csv_file_path, conca, delimiter=",", header="ID,X,Y,AOT, B1, B2, B3, B4, B5, B6, B7, B8A, B11, B12, SCL, TCI, WVP", fmt="%.6f")